[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moussa822/SoniTranslate/blob/main/SoniTranslate_Colab.ipynb)

# SoniTranslate

| Description | Link |
| ----------- | ---- |
| 🎉 Repository | [![GitHub Repository](https://img.shields.io/badge/GitHub-Repository-black?style=flat-square&logo=github)](https://github.com/R3gm/SoniTranslate/) |
| 🚀 Online Demo in HF | [![Hugging Face Spaces](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Spaces-blue)](https://huggingface.co/spaces/r3gm/SoniTranslate_translate_audio_of_a_video_content) |

In [ ]:
# @title Install requirements for SoniTranslate
%cd /content
!git clone https://github.com/moussa822/SoniTranslate.git
%cd SoniTranslate

# 1. Installation de l'outil UV
!pip uninstall chex pandas-stubs ibis-framework albumentations albucore jax -y -q
!pip install uv==0.8.13 -q
!uv venv --python 3.10 --clear -q
!curl -sS https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!uv run python get-pip.py pip==23.1.2 -q
!uv run python -m pip install -q pip==23.1.2 Setuptools==80.6.0 wheel
!apt install git-lfs -q
!git lfs install

# 2. Installation PyTorch
!uv run python -m pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124

# 3. Correction lien WhisperX
!sed -i 's|git+https://github.com/R3gm/whisperX.git@cuda_11_8|git+https://github.com/R3gm/whisperX.git@cuda_12_x|' requirements_base.txt

# 4. Installation des dependances de base
!uv run python -m pip install -q -r requirements_base.txt

# 5. Installation de Google GenAI
!uv run python -m pip install -q google-genai

# 6. Installation des extras
!uv run python -m pip install -q -r requirements_extra.txt
!uv run python -m pip install -q onnxruntime-gpu==1.22.0

# 7. Installation de Kokoro TTS
!sudo apt-get install -y espeak-ng -q
!uv run python -m pip install -q kokoro soundfile

# 8. Installation des moteurs de clonage
!uv run python -m pip install -q f5-tts
!/content/SoniTranslate/.venv/bin/pip install -q chatterbox-tts omnivoice perth grapheme s3tokenizer conformer diffusers pykakasi pyloudnorm resemble-perth spacy-pkuseg webdataset --no-deps

Install_PIPER_TTS = True  # @param {type:'boolean'}

if Install_PIPER_TTS:
    !uv run python -m pip install -q piper-tts==1.2.0

Install_Coqui_XTTS = True  # @param {type:'boolean'}

if Install_Coqui_XTTS:
    !uv run python -m pip install -q -r requirements_xtts.txt
    !uv run python -m pip install -q TTS==0.21.1 --no-deps

# 9. Installation officielle cuDNN 8 et liaison cuBLAS 12
!/content/SoniTranslate/.venv/bin/pip install -q nvidia-cudnn-cu11 nvidia-cublas-cu12
!sudo cp -f /content/SoniTranslate/.venv/lib/python3.10/site-packages/nvidia/cudnn/lib/* /usr/local/cuda/lib64/ 2>/dev/null || true
!sudo cp -f /content/SoniTranslate/.venv/lib/python3.10/site-packages/nvidia/cudnn/lib/* /usr/lib/x86_64-linux-gnu/ 2>/dev/null || true
!sudo cp -f /content/SoniTranslate/.venv/lib/python3.10/site-packages/nvidia/cublas/lib/* /usr/local/cuda/lib64/ 2>/dev/null || true
!sudo cp -f /content/SoniTranslate/.venv/lib/python3.10/site-packages/nvidia/cublas/lib/* /usr/lib/x86_64-linux-gnu/ 2>/dev/null || true
!sudo ldconfig


One important step is to accept the license agreement for using Pyannote.

Accept the licenses here:
- https://huggingface.co/pyannote/speaker-diarization
- https://huggingface.co/pyannote/segmentation

Generate your Hugging Face token here:
https://hf.co/settings/tokens

When creating the token, make sure to enable:
Read access to contents of all public gated repos you can access.

In [ ]:
#@markdown ## RUN THE WEB APP
YOUR_HF_TOKEN = ''  # @param {type:'string'}
openai_api_key = ''  # @param {type:'string'}
google_api_key = ''  # @param {type:'string'}
groq_api_key = ''  # @param {type:'string'}
elevenlabs_api_key = ''  # @param {type:'string'}

theme = 'Taithrah/Minimal'  # @param ['Taithrah/Minimal', 'aliabid94/new-theme', 'gstaff/xkcd', 'ParityError/LimeFace', 'abidlabs/pakistan', 'rottenlittlecreature/Moon_Goblin', 'ysharma/llamas', 'gradio/dracula_revamped']
interface_language = 'french'  # @param ['arabic','azerbaijani','chinese_zh_cn','english','french','german','hindi','indonesian','italian','japanese','korean','marathi','polish','portuguese','russian','spanish','swedish','turkish','ukrainian','vietnamese']
verbosity_level = 'info'  # @param ['debug','info','warning','error','critical']

import os

# ==============================================================================
# 1. INSTALLATION NATIVE DE CUDNN 8 AU NIVEAU DU SYSTEME D'EXPLOITATION
# ==============================================================================
if not os.path.exists('/usr/lib/x86_64-linux-gnu/libcudnn_ops_infer.so.8'):
    print('Telechargement et installation de cuDNN 8 dans le noyau Ubuntu...')
    os.system('wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/libcudnn8_8.9.7.29-1+cuda12.2_amd64.deb')
    os.system('sudo dpkg -i libcudnn8_8.9.7.29-1+cuda12.2_amd64.deb')
    os.system('rm libcudnn8_8.9.7.29-1+cuda12.2_amd64.deb')

# Injection directe de libcublasLt.so.12 au meme endroit
os.system('sudo cp -n /content/SoniTranslate/.venv/lib/python3.10/site-packages/nvidia/cublas/lib/* /usr/lib/x86_64-linux-gnu/ 2>/dev/null || true')
os.system('sudo ldconfig')
print('Bibliotheques Nvidia systeme injectees avec succes !')

# ==============================================================================
# 2. SYSTEME DE CACHE PERSISTANT AUTOMATIQUE AVEC GOOGLE DRIVE
# ==============================================================================
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs('/content/drive/MyDrive/SoniTranslate_Cache', exist_ok=True)
    os.system('ln -sf /content/drive/MyDrive/SoniTranslate_Cache /content/SoniTranslate/persistent_cache')
    print('Cache Persistant lie a Google Drive avec succes !')

%cd /content/SoniTranslate

# ==============================================================================
# 3. LANCEMENT DU SERVEUR AVEC LIAISON CUBLAS
# ==============================================================================
cmd_env = 'LD_LIBRARY_PATH=/content/SoniTranslate/.venv/lib/python3.10/site-packages/nvidia/cublas/lib:$LD_LIBRARY_PATH '

if YOUR_HF_TOKEN: cmd_env += 'YOUR_HF_TOKEN=' + chr(39) + YOUR_HF_TOKEN + chr(39) + ' '
if openai_api_key: cmd_env += 'OPENAI_API_KEY=' + chr(39) + openai_api_key + chr(39) + ' '
if google_api_key: cmd_env += 'GOOGLE_API_KEY=' + chr(39) + google_api_key + chr(39) + ' '
if groq_api_key: cmd_env += 'GROQ_API_KEY=' + chr(39) + groq_api_key + chr(39) + ' '
if elevenlabs_api_key: cmd_env += 'ELEVEN_API_KEY=' + chr(39) + elevenlabs_api_key + chr(39) + ' '

# Lancement isole dans l'environnement virtuel (.venv) PyTorch 2.5.1
!{cmd_env} /content/SoniTranslate/.venv/bin/python app_rvc.py --theme {theme} --verbosity_level {verbosity_level} --language {interface_language} --public_url
